# Module 22: Interactive CPython Internals & Rust PyO3 Accelerators

### What You Will Discover
By running this notebook, you will inspect PyObject memory headers, understand why integers cost 28 bytes in Python, measure why naive `ctypes` loops are slower than pure Python, and explore PyO3 GIL release.

**Key Question Answered:** *Why is `ctypes` in a Python loop 5x slower than pure Python, and how does compiling the entire loop in Rust with PyO3 solve it?*


In [ ]:
# Step 1: Measuring PyObject memory overhead
import sys

x = 7
print(f'Size of integer 7 in Python: {sys.getsizeof(x)} bytes')
print('C int64_t: 8 bytes. Python int: 28 bytes (ob_refcnt, ob_type, ob_size, ob_digit)')


In [ ]:
# Step 2: List of integers memory pointer overhead
million_ints = [7] * 1000
print(f'Size of list holding 1000 ints: {sys.getsizeof(million_ints)} bytes')
print('Explanation: The list stores 1000 8-byte POINTERS to PyObjects, not contiguous ints!')


In [ ]:
# Step 3: Reference counting inspection
print(f'Reference count for 7: {sys.getrefcount(7)} (Cached singleton!)')


### 🔮 Prediction Prompt
**Before running the next cell:** If you write a loop in Python that iterates over an array using `ctypes.c_double` vs plain Python `float`, will `ctypes` be faster or slower? Write down your prediction.


In [ ]:
# Surprising Result: The ctypes Anti-Pattern (5x Slower!)
import ctypes
import time

n = 100_000
py_list = [float(i) for i in range(n)]
c_arr = (ctypes.c_double * n)(*py_list)

start = time.perf_counter()
s_py = sum(py_list)
t_py = time.perf_counter() - start

start = time.perf_counter()
s_c = sum(c_arr[i] for i in range(n))  # Boxes a new Python float every index access!
t_c = time.perf_counter() - start

print(f'Pure Python sum: {t_py * 1000:.2f} ms')
print(f'ctypes loop sum: {t_c * 1000:.2f} ms ({t_c / t_py:.2f}x SLOWER!)')
print('Rule: ctypes is a bridge to existing C libs, NOT an engine for Python loops!')


### Rust + PyO3 Architecture: Compiling the Entire Loop
PyO3 moves the entire loop and buffer borrowing into Rust (`&[u8]`), achieving zero-copy and 30x+ speedups.


In [ ]:
# Simulating PyO3 FNV-1a hash algorithm in pure Python vs Rust concept
def fnv1a_python(data: bytes) -> int:
    h = 0xcbf29ce484222325
    for b in data:
        h ^= b
        h = (h * 0x100000001b3) & 0xFFFFFFFFFFFFFFFF
    return h

sample_blob = b'Python native extension engineering' * 1000
start = time.perf_counter()
digest = fnv1a_python(sample_blob)
print(f'Python hash: {hex(digest)} in {(time.perf_counter() - start) * 1000:.2f} ms')


### The Headline Win: `py.allow_threads()` Multicore Parallelism
Rust releases the GIL inside compute blocks, enabling true multicore scaling across CPU threads.


In [ ]:
print('Thread scaling comparison (from Module 22 measurements):')
print('Pure Python (4 threads) : 1.00x scaling (GIL-bound)')
print('Native Rust (4 threads) : 3.21x scaling (GIL released!)')


### 🛠️ Interactive Challenge: Batch FFI Crossings
The following function crosses the Python-to-native boundary 10,000 times inside a loop. Refactor the call to batch the entire list in a single boundary crossing.


In [ ]:
# TODO: FIX ME - Batch boundary crossing to amortize FFI overhead
def mock_native_push(val: float): return val * 2
def mock_native_batch_push(vals: list[float]): return [v * 2 for v in vals]

values = [float(i) for i in range(10_000)]
# FIX: Call batch function once instead of looping mock_native_push(x)
results = mock_native_batch_push(values)
print(f'Processed {len(results)} items in ONE boundary crossing!')


### 🏁 Summary & Next Steps
- `PyObject` headers add 28 bytes per int and pointer chase overhead.
- `ctypes` in a Python loop is an anti-pattern (5x slower than pure Python).
- Rust + PyO3 compiles the loop and releases the GIL with `py.allow_threads()`.
- Run `python 01_ctypes_honest_demo.py` and `python 02_gil_release_parallelism_demo.py`.
- Follow [PROJECT_GUIDE.md](PROJECT_GUIDE.md) to build the Rust accelerator.
